# Zomato Rating Predictor
## Notebook 1 — Data Cleaning & Feature Engineering

**Objective:** Clean the raw Zomato dataset and engineer features that will power the rating prediction model in Notebook 3.

**Output:** A clean, model-ready CSV saved as `zomato_clean.csv`

---
## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

# After running this cell, a link will appear. Click on it,
# select your Google account, and allow access to your Drive.

Mounted at /content/drive


Once your Drive is mounted, the `zomato.csv` file should be accessible at the path below.

---
## 1. Load Data

Upload `zomato.csv` to your Colab session before running this cell.  
You can do that via **Files (folder icon on the left) → Upload**.

In [5]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/zomato-rating-predictor/zomato.csv', encoding='latin1')

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

Shape: (9551, 21)
Columns: ['Restaurant ID', 'Restaurant Name', 'Country Code', 'City', 'Address', 'Locality', 'Locality Verbose', 'Longitude', 'Latitude', 'Cuisines', 'Average Cost for two', 'Currency', 'Has Table booking', 'Has Online delivery', 'Is delivering now', 'Switch to order menu', 'Price range', 'Aggregate rating', 'Rating color', 'Rating text', 'Votes']


,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


In [6]:
# Quick snapshot of the data types and nulls
print('=== Data Types ===')
print(df.dtypes)
print()
print('=== Missing Values ===')
print(df.isnull().sum())

=== Data Types ===
Restaurant ID             int64
Restaurant Name          object
Country Code              int64
City                     object
Address                  object
Locality                 object
Locality Verbose         object
Longitude               float64
Latitude                float64
Cuisines                 object
Average Cost for two      int64
Currency                 object
Has Table booking        object
Has Online delivery      object
Is delivering now        object
Switch to order menu     object
Price range               int64
Aggregate rating        float64
Rating color             object
Rating text              object
Votes                     int64
dtype: object

=== Missing Values ===
Restaurant ID           0
Restaurant Name         0
Country Code            0
City                    0
Address                 0
Locality                0
Locality Verbose        0
Longitude               0
Latitude                0
Cuisines                9
Average Cos

In [7]:
# Rating distribution — we need to see how many zeros there are
print('=== Aggregate Rating Distribution ===')
print(df['Aggregate rating'].value_counts().sort_index().head(15))
print()
print(f"Zero-rated restaurants (unrated, not low-rated): {(df['Aggregate rating'] == 0).sum()}")
print(f"That is {(df['Aggregate rating'] == 0).mean()*100:.1f}% of all records")

=== Aggregate Rating Distribution ===
Aggregate rating
0.0    2148
1.8       1
1.9       2
2.0       7
2.1      15
2.2      27
2.3      47
2.4      87
2.5     110
2.6     191
2.7     250
2.8     315
2.9     381
3.0     468
3.1     519
Name: count, dtype: int64

Zero-rated restaurants (unrated, not low-rated): 2148
That is 22.5% of all records


---
## 2. Drop Useless & Leaky Columns

Columns dropped and why:

| Column | Reason |
|---|---|
| `Rating color` | Derived directly from `Aggregate rating` — leakage |
| `Rating text` | Same — just a text label of the rating |
| `Restaurant ID` | Arbitrary identifier, no predictive signal |
| `Address` | Too granular and unstructured; `City` and `Locality` cover location |
| `Locality Verbose` | Redundant with `Locality` |
| `Switch to order menu` | Near-zero variance (almost all 'No') |
| `Is delivering now` | Snapshot state, not a restaurant attribute |

In [8]:
# Confirm near-zero variance on Switch to order menu
print(df['Switch to order menu'].value_counts())
print()
print(df['Is delivering now'].value_counts())

Switch to order menu
No    9551
Name: count, dtype: int64

Is delivering now
No     9517
Yes      34
Name: count, dtype: int64


In [9]:
cols_to_drop = [
    'Rating color',
    'Rating text',
    'Restaurant ID',
    'Address',
    'Locality Verbose',
    'Switch to order menu',
    'Is delivering now'
]

df = df.drop(columns=cols_to_drop)
print(f'Shape after dropping columns: {df.shape}')
print(f'Remaining columns: {df.columns.tolist()}')

Shape after dropping columns: (9551, 14)
Remaining columns: ['Restaurant Name', 'Country Code', 'City', 'Locality', 'Longitude', 'Latitude', 'Cuisines', 'Average Cost for two', 'Currency', 'Has Table booking', 'Has Online delivery', 'Price range', 'Aggregate rating', 'Votes']


---
## 3. Core Cleaning

In [10]:
# ── 3a. Remove duplicates ──────────────────────────────────────────────────
before = len(df)
df = df.drop_duplicates()
print(f'Duplicates removed: {before - len(df)}')
print(f'Shape: {df.shape}')

Duplicates removed: 0
Shape: (9551, 14)


In [11]:
# ── 3b. Handle missing Cuisines ────────────────────────────────────────────
print(f"Missing in Cuisines before: {df['Cuisines'].isnull().sum()}")
df['Cuisines'] = df['Cuisines'].fillna('Unknown')
print(f"Missing in Cuisines after:  {df['Cuisines'].isnull().sum()}")

Missing in Cuisines before: 9
Missing in Cuisines after:  0


In [12]:
# ── 3c. Remove zero-rated restaurants ─────────────────────────────────────
# Aggregate rating == 0 means the restaurant has never been rated.
# These are NOT low-rated restaurants — they are simply unrated.
# Including them would make 0 the most common target value, corrupting the model.

before = len(df)
df = df[df['Aggregate rating'] > 0].copy()
print(f'Unrated restaurants removed: {before - len(df)}')
print(f'Shape after removal: {df.shape}')
print(f'Rating range now: {df["Aggregate rating"].min()} – {df["Aggregate rating"].max()}')

Unrated restaurants removed: 2148
Shape after removal: (7403, 14)
Rating range now: 1.8 – 4.9


In [13]:
# ── 3d. Encode binary Yes/No columns ──────────────────────────────────────
binary_cols = ['Has Table booking', 'Has Online delivery']

for col in binary_cols:
    print(f'{col}: {df[col].value_counts().to_dict()}')
    df[col] = (df[col] == 'Yes').astype(int)

print()
print('After encoding:')
print(df[binary_cols].value_counts())

Has Table booking: {'No': 6292, 'Yes': 1111}
Has Online delivery: {'No': 5048, 'Yes': 2355}

After encoding:
Has Table booking  Has Online delivery
0                  0                      4370
                   1                      1922
1                  0                       678
                   1                       433
Name: count, dtype: int64


---
## 4. Feature Engineering

Six new features, each with a clear rationale.

In [14]:
# ── 4a. Country name (readable) ────────────────────────────────────────────
country_map = {
    1: 'India', 14: 'Australia', 30: 'Brazil', 37: 'Canada',
    94: 'Indonesia', 148: 'New Zealand', 162: 'Philippines',
    166: 'Qatar', 184: 'Singapore', 189: 'South Africa',
    191: 'Sri Lanka', 208: 'Turkey', 214: 'UAE',
    215: 'United Kingdom', 216: 'United States'
}

df['country_name'] = df['Country Code'].map(country_map)

# Confirm no unmapped codes
unmapped = df['country_name'].isnull().sum()
print(f'Unmapped country codes: {unmapped}')
print(df['country_name'].value_counts())

Unmapped country codes: 0
country_name
India             6513
United States      431
United Kingdom      79
UAE                 60
South Africa        60
Brazil              55
New Zealand         40
Turkey              34
Australia           24
Philippines         22
Indonesia           21
Qatar               20
Singapore           20
Sri Lanka           20
Canada               4
Name: count, dtype: int64


In [15]:
# ── 4b. Primary cuisine ────────────────────────────────────────────────────
# Most restaurants list multiple cuisines (e.g. "North Indian, Chinese, Mughlai").
# We take the first one as the primary cuisine — the one the restaurant is
# most associated with.

df['primary_cuisine'] = df['Cuisines'].str.split(',').str[0].str.strip()

print(f'Unique primary cuisines: {df["primary_cuisine"].nunique()}')
print(df['primary_cuisine'].value_counts().head(15))

Unique primary cuisines: 119
primary_cuisine
North Indian    2208
Chinese          608
Cafe             549
Fast Food        479
Bakery           432
American         270
Continental      225
Italian          218
South Indian     203
Pizza            192
Street Food      160
Mithai           145
Mughlai          128
Ice Cream        127
Desserts         124
Name: count, dtype: int64


In [16]:
# ── 4c. Cuisine count & is_multi_cuisine ───────────────────────────────────
# Hypothesis: restaurants offering more cuisine types may attract a broader
# customer base, which could influence ratings positively or negatively.

df['cuisine_count'] = df['Cuisines'].str.split(',').str.len()
df['is_multi_cuisine'] = (df['cuisine_count'] > 1).astype(int)

print('Cuisine count distribution:')
print(df['cuisine_count'].value_counts().sort_index().head(10))
print()
print(f"Multi-cuisine restaurants: {df['is_multi_cuisine'].sum()} ({df['is_multi_cuisine'].mean()*100:.1f}%)")

Cuisine count distribution:
cuisine_count
1    2233
2    2737
3    1603
4     557
5     160
6      72
7      28
8      13
Name: count, dtype: int64

Multi-cuisine restaurants: 5170 (69.8%)


In [17]:
# ── 4d. Log votes ──────────────────────────────────────────────────────────
# Votes is heavily right-skewed (a few restaurants have tens of thousands
# of votes). Log-transforming compresses the scale and makes the
# relationship with rating more linear.

print('Votes stats before transform:')
print(df['Votes'].describe())
print()

df['log_votes'] = np.log1p(df['Votes'])  # log1p handles zeros safely

print('log_votes stats:')
print(df['log_votes'].describe())

Votes stats before transform:
count     7403.000000
mean       202.185060
std        479.195199
min          4.000000
25%         19.000000
50%         60.000000
75%        181.000000
max      10934.000000
Name: Votes, dtype: float64

log_votes stats:
count    7403.000000
mean        4.158411
std         1.493421
min         1.609438
25%         2.995732
50%         4.110874
75%         5.204007
max         9.299724
Name: log_votes, dtype: float64


In [18]:
# ── 4e. Cost per person ────────────────────────────────────────────────────
# Average Cost for two divided by 2 — a more intuitive unit for modelling
# and for the Streamlit predictor inputs later.

df['cost_per_person'] = df['Average Cost for two'] / 2

print('cost_per_person stats:')
print(df['cost_per_person'].describe())

cost_per_person stats:
count      7403.000000
mean        724.207551
std        9151.762132
min           0.000000
25%         150.000000
50%         250.000000
75%         400.000000
max      400000.000000
Name: cost_per_person, dtype: float64


In [19]:
# ── 4f. Has both services ──────────────────────────────────────────────────
# Directly encodes the hypothesis we tested in the original notebook:
# restaurants with BOTH online delivery AND table booking perform better.
# Now we let the model confirm or deny this.

df['has_both_services'] = (
    (df['Has Table booking'] == 1) & (df['Has Online delivery'] == 1)
).astype(int)

print(f"Restaurants with both services: {df['has_both_services'].sum()} ({df['has_both_services'].mean()*100:.1f}%)")
print()
print('Mean rating by has_both_services:')
print(df.groupby('has_both_services')['Aggregate rating'].mean())

Restaurants with both services: 433 (5.8%)

Mean rating by has_both_services:
has_both_services
0    3.429311
1    3.612471
Name: Aggregate rating, dtype: float64


---
## 5. Encode Categorical Variables

Three categorical columns remain: `country_name`, `primary_cuisine`, and `City`.

- **`country_name`** → one-hot encoding (only 15 countries, manageable)
- **`primary_cuisine`** → frequency encoding (too many unique values for one-hot)
- **`City`** → frequency encoding (141 unique cities)

In [20]:
# ── 5a. One-hot encode country_name ───────────────────────────────────────
df = pd.get_dummies(df, columns=['country_name'], prefix='country', drop_first=False)

country_cols = [c for c in df.columns if c.startswith('country_')]
print(f'Country dummy columns created: {len(country_cols)}')
print(country_cols)

Country dummy columns created: 15
['country_Australia', 'country_Brazil', 'country_Canada', 'country_India', 'country_Indonesia', 'country_New Zealand', 'country_Philippines', 'country_Qatar', 'country_Singapore', 'country_South Africa', 'country_Sri Lanka', 'country_Turkey', 'country_UAE', 'country_United Kingdom', 'country_United States']


In [21]:
# ── 5b. Frequency encode primary_cuisine ──────────────────────────────────
# Frequency encoding replaces each category with how often it appears
# in the dataset. Rare cuisines get low values; common ones get high values.
# This preserves ordinality without exploding dimensionality.

cuisine_freq = df['primary_cuisine'].value_counts(normalize=True)
df['cuisine_freq_enc'] = df['primary_cuisine'].map(cuisine_freq)

print('Sample cuisine frequency encodings:')
print(cuisine_freq.head(10))

Sample cuisine frequency encodings:
primary_cuisine
North Indian    0.298257
Chinese         0.082129
Cafe            0.074159
Fast Food       0.064703
Bakery          0.058355
American        0.036472
Continental     0.030393
Italian         0.029448
South Indian    0.027421
Pizza           0.025935
Name: proportion, dtype: float64


In [22]:
# ── 5c. Frequency encode City ──────────────────────────────────────────────
city_freq = df['City'].value_counts(normalize=True)
df['city_freq_enc'] = df['City'].map(city_freq)

print('Top 10 city frequency encodings:')
print(city_freq.head(10))

Top 10 city frequency encodings:
City
New Delhi       0.546805
Gurgaon         0.120222
Noida           0.094016
Faridabad       0.020397
Ghaziabad       0.003107
Bhubaneshwar    0.002837
Lucknow         0.002837
Ahmedabad       0.002837
Amritsar        0.002837
Guwahati        0.002837
Name: proportion, dtype: float64


---
## 6. Final Column Selection

Drop original columns that have been replaced by engineered features,
or that we won't use in modelling.

In [23]:
cols_to_drop_final = [
    'Restaurant Name',   # identifier, not a feature
    'Country Code',      # replaced by country_name dummies
    'City',              # replaced by city_freq_enc
    'Locality',          # too granular, City covers it
    'Cuisines',          # replaced by primary_cuisine + cuisine_count
    'primary_cuisine',   # replaced by cuisine_freq_enc
    'Currency',          # collinear with country
    'Longitude',         # covered by city/country encoding
    'Latitude',
]

df = df.drop(columns=cols_to_drop_final)

print(f'Final shape: {df.shape}')
print(f'Final columns:')
for col in df.columns:
    print(f'  {col}')

Final shape: (7403, 28)
Final columns:
  Average Cost for two
  Has Table booking
  Has Online delivery
  Price range
  Aggregate rating
  Votes
  cuisine_count
  is_multi_cuisine
  log_votes
  cost_per_person
  has_both_services
  country_Australia
  country_Brazil
  country_Canada
  country_India
  country_Indonesia
  country_New Zealand
  country_Philippines
  country_Qatar
  country_Singapore
  country_South Africa
  country_Sri Lanka
  country_Turkey
  country_UAE
  country_United Kingdom
  country_United States
  cuisine_freq_enc
  city_freq_enc


---
## 7. Sanity Checks

In [24]:
# No missing values should remain
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'None — all clean.')

Missing values per column:
None — all clean.


In [25]:
# All columns should be numeric now
non_numeric = df.select_dtypes(include='object').columns.tolist()
print('Non-numeric columns remaining:')
print(non_numeric if non_numeric else 'None — all numeric.')

Non-numeric columns remaining:
None — all numeric.


In [26]:
# Target variable sanity check
print('=== Target: Aggregate rating ===')
print(df['Aggregate rating'].describe())
print()
print(f'Any zeros remaining: {(df["Aggregate rating"] == 0).sum()}')

=== Target: Aggregate rating ===
count    7403.000000
mean        3.440024
std         0.552195
min         1.800000
25%         3.000000
50%         3.400000
75%         3.800000
max         4.900000
Name: Aggregate rating, dtype: float64

Any zeros remaining: 0


In [27]:
# Feature summary
print('=== Feature Summary ===')
df.describe().T

=== Feature Summary ===


,count,mean,std,min,25%,50%,75%,max
Average Cost for two,7403.0,1448.415102,18303.524265,0.000000,300.000000,500.000000,800.000000,800000.000000
Has Table booking,7403.0,0.150074,0.357168,0.000000,0.000000,0.000000,0.000000,1.000000
Has Online delivery,7403.0,0.318114,0.465776,0.000000,0.000000,0.000000,1.000000,1.000000
Price range,7403.0,1.970147,0.930611,1.000000,1.000000,2.000000,3.000000,4.000000
Aggregate rating,7403.0,3.440024,0.552195,1.800000,3.000000,3.400000,3.800000,4.900000
Votes,7403.0,202.185060,479.195199,4.000000,19.000000,60.000000,181.000000,10934.000000
cuisine_count,7403.0,2.198568,1.134801,1.000000,1.000000,2.000000,3.000000,8.000000
is_multi_cuisine,7403.0,0.698366,0.458998,0.000000,0.000000,1.000000,1.000000,1.000000
log_votes,7403.0,4.158411,1.493421,1.609438,2.995732,4.110874,5.204007,9.299724
cost_per_person,7403.0,724.207551,9151.762132,0.000000,150.000000,250.000000,400.000000,400000.000000


---
## 8. Save Clean Dataset

In [29]:
df.to_csv('/content/drive/MyDrive/Colab Notebooks/zomato-rating-predictor/data/zomato_clean.csv', index=False)
print('Saved: zomato_clean.csv')
print(f'Final shape: {df.shape}')
print()
print('=== NOTEBOOK 1 COMPLETE ===')
print('Next: Run 02_eda.ipynb using zomato_clean.csv')

Saved: zomato_clean.csv
Final shape: (7403, 28)

=== NOTEBOOK 1 COMPLETE ===
Next: Run 02_eda.ipynb using zomato_clean.csv


---
## Summary

| Step | Action | Records/Columns |
|---|---|---|
| Raw data loaded | — | 9,551 rows × 21 cols |
| Dropped leaky/useless columns | 7 columns removed | 9,551 × 14 |
| Removed duplicates | — | Confirmed, minimal |
| Removed unrated restaurants | `Aggregate rating == 0` | ~7,400 × 14 |
| Engineered 6 new features | cuisine_count, is_multi_cuisine, log_votes, cost_per_person, has_both_services, country_name | — |
| Encoded categoricals | One-hot (country), Frequency (cuisine, city) | — |
| Final clean dataset | — | **~7,400 rows × final cols** |

> `zomato_clean.csv` is ready for EDA and modelling.